# El Quirófano de datos

## 1. Contexto y Preparación: 
La empresa quiere predecir el éxito de futuras aplicaciones móviles. Han extraído un volcado de datos de la Google Play Store, pero el pipeline de Machine Learning rechaza el archivo. Su misión como ingenieros de datos es sanear los valores faltantes aplicando reglas de negocio estrictas.

## 2. Instrucciones Técnicas: 
Abran su Jupyter Notebook. Pueden descargar el dataset googleplaystore.csv de su repositorio o usar este enlace crudo (si lo hospedas en tu GitHub):

In [102]:
import pandas as pd
import numpy as np
 
 
# Reemplaza con la URL de tu repositorio
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

df.sample(10)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
1145,"Trading 212 - Forex, Stocks, CFDs",FINANCE,4.3,37580,Varies with device,"5,000,000+",Free,0,Everyone,Finance,"August 1, 2018",4.5.2.1,4.4 and up
631,Friend Find: free chat + flirt dating app,DATING,NaN,23,11M,100+,Free,0,Mature 17+,Dating,"July 31, 2018",1.0,4.4 and up
9062,DX Simulation for X-aid Dx Belt,FAMILY,4.0,35,48M,"5,000+",Free,0,Everyone,Simulation,"August 4, 2018",1.6,4.1 and up
2595,"ooVoo Video Calls, Messaging & Stories",SOCIAL,4.3,1157004,34M,"50,000,000+",Free,0,Everyone,Social,"October 16, 2017",4.2.1,4.3 and up
2163,Mosaic puzzles,FAMILY,4.4,1595,14M,"500,000+",Free,0,Everyone,Puzzle;Brain Games,"July 4, 2018",2.8,4.0.3 and up
7215,Depths of Betrayal CE (Full),FAMILY,4.3,363,14M,"1,000+",Paid,$2.99,Everyone 10+,Casual,"August 19, 2013",1.0.0,2.3 and up
1868,Dungeon Hunter Champions: Epic Online Action RPG,GAME,4.2,26247,Varies with device,"1,000,000+",Free,0,Teen,Role Playing,"July 26, 2018",1.1.40,4.0.3 and up
3148,Skyscanner,TRAVEL_AND_LOCAL,4.5,481546,29M,"10,000,000+",Free,0,Everyone,Travel & Local,"August 6, 2018",5.48,4.4 and up
4222,imo free video calls and chat,COMMUNICATION,4.3,4785892,11M,"500,000,000+",Free,0,Everyone,Communication,"June 8, 2018",9.8.000000010501,4.0 and up
5971,Bridge Constructor Stunts FREE,FAMILY,4.1,36151,46M,"1,000,000+",Free,0,Everyone,Simulation,"April 22, 2017",2.7,3.2 and up


## 3. Preguntas de Análisis (Para responder con código y celdas Markdown):

### Pregunta A: 
Ejecuten df.isnull().sum(). Identificarán que la columna Rating tiene casi 1,500 valores nulos, mientras que Type y Android Ver tienen solo 1 o 2. ¿Por qué es arquitectónicamente seguro eliminar las filas donde falta Android Ver, pero sería un error fatal eliminar las filas donde falta Rating?

In [103]:
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [104]:
# df.groupby("Android Ver")["Android Ver"].count()

In [105]:
# df.groupby("Type")["Type"].count()

Rating es un dato de tipo MNAR (missing not at random), su ausencia nos da información, mientras que la version de android o el Type, son son o más tipo MCAR (missing completively at random), el type puede ser Free y Paid, y la versiones de android son bastantes, pero es más como un error de sistema

### Pregunta B: 
Escriban la instrucción exacta usando dropna() pasándole el parámetro subset=['Android Ver', 'Type'] para eliminar únicamente las 3 filas que tienen esos nulos. Verifiquen que el total de registros en el DataFrame apenas disminuyó.

In [106]:
print("antes: ",len(df))
dropna= df.dropna(subset=["Android Ver","Type"])
print("despues: ",len(dropna))



antes:  10841
despues:  10837


antes:  10841
despues:  10837

### Pregunta C: 
Rellenar los 1,500 nulos de Rating con el promedio global es un error, ya que las apps de la categoría FAMILY (Familia) no se califican igual que las de DATING (Citas). Utilizando la lógica de la semana pasada, escriban un bloque combinando .groupby('Category') y .transform() para rellenar los NaN de la columna Rating usando la mediana de su propia categoría.

In [107]:
# stratified imputation
df_c = df.copy()
df_c["Rating"] = df_c["Rating"].fillna(df_c.groupby("Category")["Rating"].transform("median"))
print("BEFORE: \n-------------\n",df.isnull().sum(),"\n------------------")
print("AFTER: \n-------------\n",df_c.isnull().sum())


BEFORE: 
-------------
 App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64 
------------------
AFTER: 
-------------
 App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              1
Price             0
Content Rating    1
Genres            0
Last Updated      0
Current Ver       8
Android Ver       3
dtype: int64


### Pregunta D: 
Revisen la columna Size (Tamaño de la app). Notarán que no tiene nulos matemáticos (NaN), pero muchas filas dicen "Varies with device". Para un modelo matemático, ese texto es basura indigerible. Escriban una instrucción usando df['Size'].replace(...) para convertir exactamente esa frase de texto en un nulo matemático real usando np.nan de NumPy. ¿Por qué es mejor convertirlo a NaN en esta etapa en lugar de ponerle un tamaño falso como "0 MB"?


In [108]:
df.groupby("Size")["Size"].count()

Size
1,000+                   1
1.0M                     7
1.1M                    32
1.2M                    41
1.3M                    35
                      ... 
98M                     16
992k                     1
994k                     1
99M                     39
Varies with device    1695
Name: Size, Length: 462, dtype: int64

In [110]:
df_d = df.copy()
df_d["Size"]= df_d["Size"].replace("Varies with device", np.nan)
df_d.groupby("Size")["Size"].count()

Size
1,000+     1
1.0M       7
1.1M      32
1.2M      41
1.3M      35
          ..
986k       1
98M       16
992k       1
994k       1
99M       39
Name: Size, Length: 461, dtype: int64

Ponerle un falso tamaño sería un sesgo, es mejor usar NaN para poder hacerle skip más facil los datos.